# Neuroevolution (NEAT) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Evolve neural networks by protecting innovations while topology becomes more complex
NEAT is neuroevolution with historical markings. It starts simple, mutates weights and structure, and uses innovation numbers plus speciation so a new hidden node is not eliminated before it has a chance to become useful.

In [ ]:
# 1) Weight mutation changes a network's behavior
x=np.array([2.,1.]); y=1.0; w=np.array([1.0,-0.5]); pred=w@x; mse=(pred-y)**2; mut=w+np.array([0.2,0.1]); mpred=mut@x; mmse=(mpred-y)**2
plt.figure(figsize=(6,3)); plt.bar(['before','after'],[mse,mmse]); plt.ylabel('squared error'); plt.title('weight mutation can help or hurt')
assert abs(pred-1.5)<1e-12 and abs(mse-0.25)<1e-12 and abs(mmse-1.0)<1e-12

In [ ]:
# 2) Adding a connection increases expressivity
xs=np.linspace(-2,2,80); y=xs**2; linear=0.5*xs; with_skip=0.5*xs+0.25*xs**2
plt.figure(figsize=(6,3)); plt.plot(xs,y,'k--',label='target'); plt.plot(xs,linear,label='old network'); plt.plot(xs,with_skip,label='new connection adds curvature'); plt.legend(); plt.title('structural mutation changes the function class')
assert np.mean((with_skip-y)**2) < np.mean((linear-y)**2)

In [ ]:
# 3) Innovation numbers align matching genes during crossover
parentA={1:0.5,2:-1.0,4:0.7}; parentB={1:0.6,3:1.2,4:0.4}; matching=sorted(set(parentA)&set(parentB)); disjoint=sorted(set(parentA)^set(parentB))
plt.figure(figsize=(6,3)); plt.bar(['matching','disjoint/excess'],[len(matching),len(disjoint)]); plt.title('historical markings say which genes correspond')
assert matching==[1,4] and disjoint==[2,3]

In [ ]:
# 4) Speciation distance keeps novel topologies comparable with peers
Wbar=abs(parentA[1]-parentB[1])+abs(parentA[4]-parentB[4]); Wbar/=2; E=0; D=len(disjoint); N=max(len(parentA),len(parentB)); c1=c2=1.; c3=0.4; delta=(c1*E+c2*D)/N+c3*Wbar
plt.figure(figsize=(6,3)); plt.bar(['disjoint/N','0.4*weight gap','delta'],[D/N,0.4*Wbar,delta]); plt.title('compatibility distance mixes topology and weights')
assert abs(delta-0.7466666666666666)<1e-12

In [ ]:
# 5) A toy NEAT-like population benefits from structural mutations
rng=np.random.default_rng(12); xs=np.linspace(-1,1,40); y=xs**2; pop=[(rng.normal(),0.0,False) for _ in range(20)]; best=[]; hidden_counts=[]
for _ in range(18):
    errs=np.array([np.mean(((a*xs+(b*xs**2 if h else 0))-y)**2) for a,b,h in pop]); best.append(errs.min()); hidden_counts.append(sum(h for _,_,h in pop))
    probs=softmax_min(errs, temp=0.2); new=[]
    for _ in range(20):
        a,b,h=pop[rng.choice(len(pop),p=probs)]; a+=rng.normal(0,0.15); b+=rng.normal(0,0.15)
        if (not h) and rng.random()<0.2: h=True; b=rng.normal(0.5,0.2)
        new.append((a,b,h))
    pop=new
plt.figure(figsize=(6,3)); plt.plot(best,label='best error'); plt.twinx(); plt.plot(hidden_counts,color='tab:orange',label='hidden-enabled count'); plt.title('complexity grows only when selected')
assert best[-1] < best[0] and hidden_counts[-1] >= hidden_counts[0]